# **Notebook 4: Model Development and Hyperparameter Tuning**

### **Import libraries and load the NB3 fold-specific data**

In [1]:
import json, warnings, time, joblib
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (fbeta_score, recall_score, roc_auc_score, balanced_accuracy_score,
                             precision_score, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline as SkPipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

from scipy.stats import norm

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
OUT_DIR = PROJECT_ROOT / "NB_OUTS1_TRAINONLY"

assert (OUT_DIR / "NB3_foldsafe_manifest.json").exists(), \
    "NB3 outputs not found. Run NB3 first."

FOLD_DIR=OUT_DIR/'folds'
with open(OUT_DIR/'NB3_foldsafe_manifest.json') as f:
    manifest=json.load(f)
assert len(manifest['folds'])==5,'Expected five folds from updated NB3.'

lr_train = pd.read_csv(OUT_DIR/'NB3_LR_lasso_train.csv',index_col='orig_index')
rf_train = pd.read_csv(OUT_DIR/'NB3_RandomForest_train.csv',index_col='orig_index')
ann_train = pd.read_csv(OUT_DIR/'NB3_ANN_train.csv',index_col='orig_index')
y_train = lr_train['depressed_flag'].astype(int)
assert y_train.equals(rf_train['depressed_flag']) and y_train.equals(ann_train['depressed_flag'])
assert len(y_train)==530
selected={'lr':[c for c in lr_train if c!='depressed_flag'],
          'rf':[c for c in rf_train if c!='depressed_flag'],
          'ann':[c for c in ann_train if c!='depressed_flag']}
print('Fold-safe data:',OUT_DIR)
print('Final selected predictor counts:',{k:len(v) for k,v in selected.items()})
print('Fold-specific feature sets are used for cross-validation.')

Fold-safe data: /home/kathleen/THESIS1/NB_OUTS1_TRAINONLY
Final selected predictor counts: {'lr': 18, 'rf': 68, 'ann': 67}
Fold-specific feature sets are used for cross-validation.


### **Cross-validation folds and model specifications**

In [2]:
def load_fold(model, fold_record):
    fn = fold_record['files'][model]
    tr=pd.read_csv(FOLD_DIR/fn['train'],index_col='orig_index')
    val=pd.read_csv(FOLD_DIR/fn['val'],index_col='orig_index')
    assert set(tr.index).isdisjoint(set(val.index)),'Overlapping fold training/validation rows!'
    cols=[c for c in tr.columns if c!='depressed_flag']
    assert cols==[c for c in val.columns if c!='depressed_flag']
    assert not tr[cols].isna().any().any() and not val[cols].isna().any().any()
    return (tr[cols],tr['depressed_flag'].astype(int),
            val[cols],val['depressed_flag'].astype(int))

for model in ['lr','rf','ann']:
    val_indices=[]
    for fd in manifest['folds']:
        xtr,ytr,xval,yval=load_fold(model,fd)
        assert len(xtr)==fd['n_train'] and len(xval)==fd['n_val']
        assert ytr.equals(y_train.loc[xtr.index]) and yval.equals(y_train.loc[xval.index])
        val_indices.extend(xval.index.tolist())
    assert len(val_indices)==530 and len(set(val_indices))==530
    assert set(val_indices)==set(y_train.index)
print('PASS: all fold labels and row identities align with the original training data.')

def build_lr(c, tuned):
    return LogisticRegression(C=c, l1_ratio=1.0 if tuned else 0.0,
          solver='saga' if tuned else 'lbfgs', class_weight='balanced',
          random_state=RANDOM_STATE, max_iter=10000, tol=1e-3)

def build_rf(params=None):
    cfg={'n_estimators':200,'class_weight':'balanced','random_state':RANDOM_STATE,'n_jobs':-1}
    if params is not None:cfg.update(params)
    return RandomForestClassifier(**cfg)

def build_ann(params=None):
    cfg={'max_iter':2000,'random_state':RANDOM_STATE}
    if params is not None:cfg.update(params)
    return ImbPipeline([
        ('scaler',StandardScaler()),
        ('oversample',RandomOverSampler(random_state=RANDOM_STATE)),
        ('mlp',MLPClassifier(**cfg))
    ])


FOLD_TRAIN_PROBAS = {}

def eval_oof_folds(model_key, make_model, model_label):
    """All fold fitting happens on that fold's internally selected features only."""
    probs=pd.Series(index=y_train.index,dtype='float64')
    pred05=pd.Series(index=y_train.index,dtype='int64')
    score_rows=[]
    for rec in manifest['folds']:
        Xtr,ytr,Xv,yv = load_fold(model_key,rec)
        estimator=make_model(rec)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',ConvergenceWarning)
            estimator.fit(Xtr,ytr)
        FOLD_TRAIN_PROBAS[(model_label, rec['fold'])] = (ytr.to_numpy(), estimator.predict_proba(Xtr)[:,1])
        p=estimator.predict_proba(Xv)[:,1]
        yp=(p>=0.5).astype(int)
        probs.loc[Xv.index]=p
        pred05.loc[Xv.index]=yp
        score_rows.append({'fold':rec['fold'],'recall':recall_score(yv,yp,zero_division=0),
                           'auc':roc_auc_score(yv,p),'n_features':Xtr.shape[1]})
    assert not probs.isna().any()
    return probs, pd.DataFrame(score_rows)

def f2_optimal_threshold(oof_probs, y):
    grid=np.linspace(.01,.99,99)
    scores=[fbeta_score(y, np.asarray(oof_probs)>=t,beta=2,zero_division=0) for t in grid]
    return float(grid[int(np.argmax(scores))])

def metrics(ytrue, prob, thr):
    yp=(np.asarray(prob)>=thr).astype(int)
    return {'recall':recall_score(ytrue,yp,zero_division=0),
            'precision':precision_score(ytrue,yp,zero_division=0),
            'balanced_accuracy':balanced_accuracy_score(ytrue,yp),
            'auc':roc_auc_score(ytrue,prob),
            'confusion_matrix':confusion_matrix(ytrue,yp).tolist()}

PASS: all fold labels and row identities align with the original training data.


### **Model 1: Logistic Regression — baseline and tuned (LASSO)**

In [3]:
LR_C=float(pd.read_csv(OUT_DIR/'NB3_LR_best_C.csv')['best_c'].iloc[0])

def lr_cv_factory(rec, tuned):
    c=rec['best_c'] if tuned else 1.0
    # Pipeline fits StandardScaler only on this fold's training rows.
    from sklearn.pipeline import Pipeline
    return Pipeline([('scaler',StandardScaler()),('lr',build_lr(c,tuned))])

lr_base_oof, lr_base_folds=eval_oof_folds('lr', lambda rec:lr_cv_factory(rec,False), 'LR_baseline')
lr_tuned_oof, lr_tuned_folds=eval_oof_folds('lr',lambda rec:lr_cv_factory(rec,True), 'LR_tuned')
LR_THRESHOLD=f2_optimal_threshold(lr_tuned_oof,y_train)
print('LR fold-level baseline 0.5 recall:',lr_base_folds['recall'].round(3).tolist())
print('LR fold-level tuned 0.5 recall   :',lr_tuned_folds['recall'].round(3).tolist())
print(f'Full-data C={LR_C}, fold-safe tuned LR F2 threshold={LR_THRESHOLD:.2f}')

LR fold-level baseline 0.5 recall: [0.667, 0.667, 0.375, 0.562, 0.438]
LR fold-level tuned 0.5 recall   : [0.733, 0.733, 0.375, 0.5, 0.5]
Full-data C=0.06, fold-safe tuned LR F2 threshold=0.36


### **Model 2: Random Forest — baseline and recall-tuned**

In [4]:
RF_GRID={'n_estimators':[200,400],
         'max_depth':[5,8,12],
         'min_samples_leaf':[10,20,30,40],
         'max_features':['sqrt',0.5]}

def evaluate_grid(model_key, param_grid, model_builder, tag):
    start=time.time()
    rows=[]
    for num,params in enumerate(ParameterGrid(param_grid),start=1):
        per_fold=[]
        for rec in manifest['folds']:
            Xtr,ytr,Xv,yv=load_fold(model_key,rec)
            model=model_builder(params)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore',ConvergenceWarning)
                model.fit(Xtr,ytr)
            p=model.predict_proba(Xv)[:,1]
            per_fold.append(recall_score(yv,(p>=0.5).astype(int),zero_division=0))
        rows.append({'candidate_order':num,**params,'mean_test_score':float(np.mean(per_fold)),
                     'std_test_score':float(np.std(per_fold)),
                     **{f'split{i+1}_test_score':s for i,s in enumerate(per_fold)}})
        if num%8==0 or num==len(ParameterGrid(param_grid)):
            print(f'{tag}: {num}/{len(ParameterGrid(param_grid))} candidates checked ({time.time()-start:.0f}s)',flush=True)
    frame=pd.DataFrame(rows).sort_values(['mean_test_score','candidate_order'],ascending=[False,True]).reset_index(drop=True)
    best={k:frame.loc[0,k] for k in param_grid}
    for k,v in best.items():
        if isinstance(v,np.integer):best[k]=int(v)
        elif isinstance(v,np.floating):best[k]=float(v)
    print(f'{tag} best params: {best} | mean fold-safe CV recall={frame.loc[0,"mean_test_score"]:.4f}')
    return best,frame

RF_BEST,rf_grid_results=evaluate_grid('rf',RF_GRID,build_rf,'RF')
rf_grid_results.to_csv(OUT_DIR/'NB4_RF_grid_results.csv',index=False)
rf_base_oof,rf_base_folds=eval_oof_folds('rf',lambda rec:build_rf(), 'RF_baseline')
rf_tuned_oof,rf_tuned_folds=eval_oof_folds('rf',lambda rec:build_rf(RF_BEST), 'RF_tuned')
RF_THRESHOLD=f2_optimal_threshold(rf_tuned_oof,y_train)
print(f'Fold-safe RF tuned F2 threshold={RF_THRESHOLD:.2f}')

RF: 8/48 candidates checked (38s)
RF: 16/48 candidates checked (80s)
RF: 24/48 candidates checked (115s)
RF: 32/48 candidates checked (148s)
RF: 40/48 candidates checked (181s)
RF: 48/48 candidates checked (223s)
RF best params: {'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 40, 'max_features': 0.5} | mean fold-safe CV recall=0.7183
Fold-safe RF tuned F2 threshold=0.44


### **Model 3: Artificial Neural Network — baseline and recall-tuned**

In [5]:
ANN_GRID={'hidden_layer_sizes':[(4,),(8,),(16,),(32,16)],
          'alpha':[0.001,0.01,0.1,1,3,10]}
ANN_BEST,ann_grid_results=evaluate_grid('ann',ANN_GRID,build_ann,'ANN')
ann_grid_results.to_csv(OUT_DIR/'NB4_ANN_grid_results.csv',index=False)
ann_base_oof,ann_base_folds=eval_oof_folds('ann',lambda rec:build_ann(), 'ANN_baseline')
ann_tuned_oof,ann_tuned_folds=eval_oof_folds('ann',lambda rec:build_ann(ANN_BEST), 'ANN_tuned')
ANN_THRESHOLD=f2_optimal_threshold(ann_tuned_oof,y_train)
print(f'Fold-safe ANN tuned F2 threshold={ANN_THRESHOLD:.2f}')

ANN: 8/24 candidates checked (38s)
ANN: 16/24 candidates checked (89s)
ANN: 24/24 candidates checked (134s)
ANN best params: {'hidden_layer_sizes': (4,), 'alpha': 0.01} | mean fold-safe CV recall=0.5425
Fold-safe ANN tuned F2 threshold=0.25


### **Cross-validation results**

In [6]:
cv_scores=[]
for nm,df in [('LR_baseline',lr_base_folds),('LR_tuned',lr_tuned_folds),
              ('RF_baseline',rf_base_folds),('RF_tuned',rf_tuned_folds),
              ('ANN_baseline',ann_base_folds),('ANN_tuned',ann_tuned_folds)]:
    for _,r in df.iterrows():
        cv_scores.append({'model':nm,**r.to_dict()})
pd.DataFrame(cv_scores).to_csv(OUT_DIR/'NB4_foldsafe_CV_fold_metrics.csv',index=False)
for nm,probs in [('LR_baseline',lr_base_oof),('LR_tuned',lr_tuned_oof),
                 ('RF_baseline',rf_base_oof),('RF_tuned',rf_tuned_oof),
                 ('ANN_baseline',ann_base_oof),('ANN_tuned',ann_tuned_oof)]:
    assert probs.index.equals(y_train.index)
oof=pd.DataFrame({'orig_index':y_train.index,'y_true':y_train.values,
                  'LR_baseline_proba':lr_base_oof.values,'LR_tuned_proba':lr_tuned_oof.values,
                  'RF_baseline_proba':rf_base_oof.values,'RF_tuned_proba':rf_tuned_oof.values,
                  'ANN_baseline_proba':ann_base_oof.values,'ANN_tuned_proba':ann_tuned_oof.values})
oof.to_csv(OUT_DIR/'NB4_foldsafe_OOF_predictions.csv',index=False)
print('Saved fold-safe CV folds and out-of-fold training probabilities.')

Saved fold-safe CV folds and out-of-fold training probabilities.


### **Training-validation performance summary**

In [7]:
MODEL_ORDER = ['LR_baseline', 'LR_tuned', 'RF_baseline', 'RF_tuned', 'ANN_baseline', 'ANN_tuned']
OOF_BY_MODEL = {
    'LR_baseline': lr_base_oof,
    'LR_tuned': lr_tuned_oof,
    'RF_baseline': rf_base_oof,
    'RF_tuned': rf_tuned_oof,
    'ANN_baseline': ann_base_oof,
    'ANN_tuned': ann_tuned_oof,
}
THRESHOLDS = {
    'LR_baseline': 0.50,
    'LR_tuned': LR_THRESHOLD,
    'RF_baseline': 0.50,
    'RF_tuned': RF_THRESHOLD,
    'ANN_baseline': 0.50,
    'ANN_tuned': ANN_THRESHOLD,
}

summary_rows = []
for name in MODEL_ORDER:
    probabilities = OOF_BY_MODEL[name].loc[y_train.index].to_numpy()
    threshold = THRESHOLDS[name]
    train_recall = []
    val_recall = []
    family = name.split('_')[0].lower()

    for fold in manifest['folds']:
        fold_id = fold['fold']
        fold_y, fold_p = FOLD_TRAIN_PROBAS[(name, fold_id)]
        train_recall.append(recall_score(
            fold_y, (fold_p >= threshold).astype(int), zero_division=0
        ))
        _, _, x_val, y_val = load_fold(family, fold)
        val_prob = OOF_BY_MODEL[name].loc[x_val.index].to_numpy()
        val_recall.append(recall_score(
            y_val, (val_prob >= threshold).astype(int), zero_division=0
        ))

    summary_rows.append({
        'model': name,
        'threshold': float(threshold),
        'cv_recall': float(recall_score(
            y_train, (probabilities >= threshold).astype(int), zero_division=0
        )),
        'cv_auc': float(roc_auc_score(y_train, probabilities)),
        'cv_balanced_accuracy': float(balanced_accuracy_score(
            y_train, (probabilities >= threshold).astype(int)
        )),
        'cv_train_recall_mean': float(np.mean(train_recall)),
        'cv_val_recall_mean': float(np.mean(val_recall)),
        'cv_recall_gap': float(abs(np.mean(train_recall) - np.mean(val_recall))),
    })

cv_summary = pd.DataFrame(summary_rows)
cv_summary.to_csv(OUT_DIR / 'NB4_training_cv_summary.csv', index=False)
print('=== TRAINING CROSS-VALIDATION PERFORMANCE ===')
print(cv_summary.round(4).to_string(index=False))

=== TRAINING CROSS-VALIDATION PERFORMANCE ===
       model  threshold  cv_recall  cv_auc  cv_balanced_accuracy  cv_train_recall_mean  cv_val_recall_mean  cv_recall_gap
 LR_baseline       0.50     0.5385  0.7396                0.6841                0.8494              0.5417         0.3077
    LR_tuned       0.36     0.8077  0.7646                0.7324                0.9422              0.8108         0.1314
 RF_baseline       0.50     0.2564  0.7992                0.6028                1.0000              0.2592         0.7408
    RF_tuned       0.44     0.7821  0.7705                0.7151                0.8620              0.7833         0.0787
ANN_baseline       0.50     0.3846  0.7679                0.6492                1.0000              0.3883         0.6117
   ANN_tuned       0.25     0.6410  0.7063                0.6911                0.9905              0.6442         0.3463


### **Final model fitting and saved outputs**

In [8]:
final_models={
 'LR_baseline':SkPipeline([('scaler',StandardScaler()),('lr',build_lr(1.0,False))]),
 'LR_tuned':SkPipeline([('scaler',StandardScaler()),('lr',build_lr(LR_C,True))]),
 'RF_baseline':build_rf(),
 'RF_tuned':build_rf(RF_BEST),
 'ANN_baseline':build_ann(),
 'ANN_tuned':build_ann(ANN_BEST)
}
full_train={'LR':lr_train,'RF':rf_train,'ANN':ann_train}
feature_cols={'LR':selected['lr'],'RF':selected['rf'],'ANN':selected['ann']}
for name,est in final_models.items():
    fam=name.split('_')[0]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',ConvergenceWarning)
        est.fit(full_train[fam][feature_cols[fam]], y_train)
    joblib.dump(est,OUT_DIR/f'NB4_fitted_{name}.joblib')
    print('Fitted and saved:',name,'predictors=',len(feature_cols[fam]))

lr_model=final_models['LR_tuned'].named_steps['lr']
lr_coef_df=pd.DataFrame({'feature':selected['lr'],'coefficient':lr_model.coef_[0]})
lr_coef_df=lr_coef_df.sort_values('coefficient',key=lambda s:s.abs(),ascending=False)
lr_coef_df.to_csv(OUT_DIR/'NB4_LR_coefficients.csv',index=False)
rf_imp_df=pd.DataFrame({'feature':selected['rf'],
                         'importance':final_models['RF_tuned'].feature_importances_}).sort_values('importance',ascending=False)
rf_imp_df['cumulative_importance']=rf_imp_df['importance'].cumsum()
rf_imp_df.to_csv(OUT_DIR/'NB4_RF_feature_importances.csv',index=False)

hp=[]
for name,t in THRESHOLDS.items():hp.append({'model':name,'parameter':'threshold','value':str(t)})
hp.append({'model':'LR_tuned','parameter':'C','value':str(LR_C)})
for name,parameters in [('RF_tuned',RF_BEST),('ANN_tuned',ANN_BEST)]:
    for k,v in parameters.items():hp.append({'model':name,'parameter':k,'value':str(v)})
pd.DataFrame(hp).to_csv(OUT_DIR/'NB4_best_hyperparameters.csv',index=False)
print('\nSaved six fitted models and their hyperparameters.')

Fitted and saved: LR_baseline predictors= 18
Fitted and saved: LR_tuned predictors= 18
Fitted and saved: RF_baseline predictors= 68
Fitted and saved: RF_tuned predictors= 68
Fitted and saved: ANN_baseline predictors= 67
Fitted and saved: ANN_tuned predictors= 67

Saved six fitted models and their hyperparameters.


### **Model summaries and coefficient tables**

In [9]:
models = MODEL_ORDER
rows = []

for name in models:
    family = name.split('_')[0]
    est = final_models[name]
    r = cv_summary.set_index('model').loc[name]
    rec = dict(model=name, train_rows=len(y_train), predictors=len(selected[family.lower()]),
               decision_threshold=float(r['threshold']), cv_recall=float(r['cv_recall']),
               cv_auc=float(r['cv_auc']), cv_balanced_accuracy=float(r['cv_balanced_accuracy']),
               cv_train_recall=float(r['cv_train_recall_mean']),
               cv_validation_recall=float(r['cv_val_recall_mean']),
               cv_recall_gap=float(r['cv_recall_gap']))
    if family=='LR':
        lr = est.named_steps['lr']; rec['hyperparameters']=f"C={lr.C}; penalty={'L1' if name.endswith('tuned') else 'L2'}"
    elif family=='RF':
        rec['hyperparameters']=f"trees={est.n_estimators}; depth={est.max_depth}; leaf={est.min_samples_leaf}; max_features={est.max_features}"
    else:
        mlp = est.named_steps['mlp']; rec['hyperparameters']=f"layers={mlp.hidden_layer_sizes}; alpha={mlp.alpha}; iterations={mlp.n_iter_}"
    rows.append(rec)
    print('\n'+'='*64)
    print(name.upper()+' | MODEL CARD (TRAINING CROSS-VALIDATION)')
    print('='*64)
    for k,v in rec.items():
        print(f'  {k:<24}: {v:.4f}' if isinstance(v, (float,np.floating)) else f'  {k:<24}: {v}')
model_cards = pd.DataFrame(rows)
model_cards.to_csv(OUT_DIR/'NB4_training_model_cards.csv',index=False)
print('\nTRAINING CV MODEL COMPARISON')
print(model_cards[['model','decision_threshold','cv_recall','cv_auc','cv_balanced_accuracy','cv_recall_gap']].round(4).to_string(index=False))


LR_BASELINE | MODEL CARD (TRAINING CROSS-VALIDATION)
  model                   : LR_baseline
  train_rows              : 530
  predictors              : 18
  decision_threshold      : 0.5000
  cv_recall               : 0.5385
  cv_auc                  : 0.7396
  cv_balanced_accuracy    : 0.6841
  cv_train_recall         : 0.8494
  cv_validation_recall    : 0.5417
  cv_recall_gap           : 0.3077
  hyperparameters         : C=1.0; penalty=L2

LR_TUNED | MODEL CARD (TRAINING CROSS-VALIDATION)
  model                   : LR_tuned
  train_rows              : 530
  predictors              : 18
  decision_threshold      : 0.3600
  cv_recall               : 0.8077
  cv_auc                  : 0.7646
  cv_balanced_accuracy    : 0.7324
  cv_train_recall         : 0.9422
  cv_validation_recall    : 0.8108
  cv_recall_gap           : 0.1314
  hyperparameters         : C=0.06; penalty=L1

RF_BASELINE | MODEL CARD (TRAINING CROSS-VALIDATION)
  model                   : RF_baseline
  train_rows   

### **LR coefficients and RF feature importance**

In [10]:
lr_model = final_models['LR_tuned']
assert list(lr_model.named_steps.keys()) == ['scaler','lr']
features = selected['lr']
scaler = lr_model.named_steps['scaler']
coefs = lr_model.named_steps['lr'].coef_[0]
assert len(features)==len(coefs)
coef_df = pd.DataFrame({'feature':features,
                        'coefficient_standardized':coefs,
                        'odds_ratio_per_1SD':np.exp(np.clip(coefs,-50,50)),
                        'training_mean':scaler.mean_,
                        'training_scale':scaler.scale_,
                        'coefficient_per_original_unit':coefs/scaler.scale_})
coef_df['odds_ratio_per_original_unit']=np.exp(np.clip(coef_df['coefficient_per_original_unit'],-50,50))
coef_df=coef_df.reindex(coef_df['coefficient_standardized'].abs().sort_values(ascending=False).index)
print('TUNED L1 LOGISTIC REGRESSION - COEFFICIENTS')
print(coef_df[['feature','coefficient_standardized','odds_ratio_per_1SD','odds_ratio_per_original_unit']].round(4).to_string(index=False))
print('The standardized odds ratio applies to a one-standard-deviation increase.')
print('Standard errors and classical Wald p-values are not available for this regularized fit.')
coef_df.to_csv(OUT_DIR/'NB4_LR_coefficients_detailed.csv', index=False)

rf=final_models['RF_tuned']
rf_df=pd.DataFrame({'feature':selected['rf'],'importance':rf.feature_importances_}).sort_values('importance',ascending=False)
rf_df['cumulative_importance']=rf_df.importance.cumsum()
print('\nTUNED RF FEATURE IMPORTANCE (TOP 25)')
print(rf_df.head(25).round(4).to_string(index=False))
rf_df.to_csv(OUT_DIR/'NB4_RF_feature_importances.csv',index=False)

TUNED L1 LOGISTIC REGRESSION - COEFFICIENTS
                          feature  coefficient_standardized  odds_ratio_per_1SD  odds_ratio_per_original_unit
                         se_score                   -0.6462              0.5240                        0.8285
                  C1I_3. Disagree                   -0.3177              0.7278                        0.5205
                               C5                   -0.3155              0.7295                        0.8532
D8G_Every day or almost every day                    0.3107              1.3644                        3.0484
   D37_PSOC_Personal care workers                    0.2596              1.2963                       68.9357
                D10_Yes, academic                   -0.2473              0.7809                        0.6096
                             I102                    0.2341              1.2638                        1.1892
                          A19_Yes                    0.2187              1.2

### **Hyperparameter comparison tables**

In [11]:
for family in ('RF','ANN'):
    grid_file = OUT_DIR / f'NB4_{family}_grid_results.csv'
    if not grid_file.exists():
        print(f'{family} grid search CSV not found.')
        continue
    grid = pd.read_csv(grid_file)
    print(f'\n{family} TOP 10 HYPERPARAMETER CONFIGURATIONS')
    print(grid.sort_values('mean_test_score',ascending=False).head(10).round(4).to_string(index=False))
print('\nSelected settings and thresholds')
print(pd.read_csv(OUT_DIR/'NB4_best_hyperparameters.csv').to_string(index=False))


RF TOP 10 HYPERPARAMETER CONFIGURATIONS
 candidate_order  max_depth max_features  min_samples_leaf  n_estimators  mean_test_score  std_test_score  split1_test_score  split2_test_score  split3_test_score  split4_test_score  split5_test_score
              15          5          0.5                40           200           0.7183          0.0882             0.8667             0.6000             0.6875             0.7500             0.6875
              31          8          0.5                40           200           0.7183          0.0882             0.8667             0.6000             0.6875             0.7500             0.6875
              47         12          0.5                40           200           0.7183          0.0882             0.8667             0.6000             0.6875             0.7500             0.6875
              16          5          0.5                40           400           0.7058          0.0873             0.8667             0.6000            

### **Logistic Regression predictor correlations**

In [12]:
X_lr_vif = lr_train[selected['lr']].astype(float)
X_lr_scaled = pd.DataFrame(
    StandardScaler().fit_transform(X_lr_vif),
    index=X_lr_vif.index, columns=X_lr_vif.columns
)
X_vif_constant = sm.add_constant(X_lr_scaled, has_constant='add')
vif_rows = []
for j, name in enumerate(X_vif_constant.columns):
    if name == 'const':
        continue
    with np.errstate(divide='ignore', invalid='ignore'):
        value = variance_inflation_factor(X_vif_constant.to_numpy(), j)
    vif_rows.append({'feature': name, 'VIF': value})
lr_vif = pd.DataFrame(vif_rows).sort_values('VIF', ascending=False)
lr_vif['flag'] = np.select(
    [lr_vif.VIF > 10, lr_vif.VIF > 5],
    ['high (>10)', 'moderate (>5)'], default='ok'
)
print('=== LR VIF (TRAINING PREDICTORS) ===')
print(lr_vif.round(2).to_string(index=False))
print('Condition number:', round(float(np.linalg.cond(X_lr_scaled)), 2))
lr_vif.to_csv(OUT_DIR/'NB4_LR_VIF.csv', index=False)

corr = X_lr_scaled.corr()
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack().reset_index()
pairs.columns = ['feature_a', 'feature_b', 'correlation']
pairs = pairs.loc[pairs['correlation'].abs() >= .5].sort_values(
    'correlation', key=lambda s: s.abs(), ascending=False
)
print('\n=== LR FEATURE PAIRS (ABS CORRELATION >= 0.50) ===')
print('None' if pairs.empty else pairs.round(3).to_string(index=False))
pairs.to_csv(OUT_DIR/'NB4_LR_correlated_pairs.csv', index=False)

=== LR VIF (TRAINING PREDICTORS) ===
                          feature  VIF flag
                         se_score 1.37   ok
                               C5 1.22   ok
D8G_Every day or almost every day 1.19   ok
D8F_Every day or almost every day 1.16   ok
                             I102 1.16   ok
                  C1G_3. Disagree 1.15   ok
                  C1I_3. Disagree 1.13   ok
                        D8F_Never 1.11   ok
          G95_Yes, more than once 1.10   ok
         E11Q_Several times a day 1.07   ok
                          E14_Yes 1.06   ok
                D10_Yes, academic 1.05   ok
             G32_Smoking more now 1.05   ok
             B31F_Once in a while 1.05   ok
   D37_PSOC_Personal care workers 1.04   ok
                          A19_Yes 1.03   ok
                    B8A_1_Grade 3 1.02   ok
  D37_PSOC_Teaching professionals 1.02   ok
Condition number: 1.93

=== LR FEATURE PAIRS (ABS CORRELATION >= 0.50) ===
None


### **Additional LR coefficient diagnostics**

In [13]:
RUN_EXPLORATORY_FIRTH = True
if RUN_EXPLORATORY_FIRTH:
    from scipy.special import expit
    Xraw = lr_train[selected['lr']].to_numpy(dtype=float)
    xs = StandardScaler().fit_transform(Xraw)
    design = np.column_stack([np.ones(len(xs)), xs])
    y = y_train.to_numpy(dtype=float)

    def firth_refit(X, y, max_iter=200, tol=1e-8):
        beta = np.zeros(X.shape[1], dtype=float)
        converged=False
        for it in range(1,max_iter+1):
            p=expit(np.clip(X@beta,-30,30)); w=np.maximum(p*(1-p),1e-12)
            inverse=np.linalg.pinv(X.T@(X*w[:,None]))
            h=w*np.einsum('ij,jk,ik->i',X,inverse,X)
            step=inverse@(X.T@(y-p+h*(0.5-p)))
            mx=np.max(np.abs(step))
            if mx>5:step=step*(5/mx)
            beta+=step
            if np.max(np.abs(step))<tol:
                converged=True
                break
        p=expit(np.clip(X@beta,-30,30)); w=np.maximum(p*(1-p),1e-12)
        cov=np.linalg.pinv(X.T@(X*w[:,None]))
        se=np.sqrt(np.clip(np.diag(cov),0,np.inf))
        return beta,se,converged,it

    beta,se,converged,niter=firth_refit(design,y)
    diag=pd.DataFrame({'feature':['intercept']+selected['lr'],
                       'firth_coefficient':beta,'approx_std_error':se})
    diag['approx_z']=np.divide(beta,se,out=np.full(beta.shape,np.nan),where=se>0)
    diag['approx_p_value']=2*norm.sf(np.abs(diag['approx_z'])) if converged else np.nan
    diag['odds_ratio']=np.exp(np.clip(beta,-50,50))
    if converged:
        diag.loc[1:,'p_adj_BH']=multipletests(diag.loc[1:,'approx_p_value'],method='fdr_bh')[1]
    print(f'EXPLORATORY FIRTH REFIT - converged={converged}, iterations={niter}')
    print('This table belongs to a separate fit without L1 regularization or class weights.')
    print('P-values are approximate and post-selection; do not use as confirmatory significance tests.')
    print(diag.round(4).to_string(index=False))
    diag.to_csv(OUT_DIR/'NB4_LR_exploratory_Firth_diagnostics.csv',index=False)

EXPLORATORY FIRTH REFIT - converged=True, iterations=24
This table belongs to a separate fit without L1 regularization or class weights.
P-values are approximate and post-selection; do not use as confirmatory significance tests.
                          feature  firth_coefficient  approx_std_error  approx_z  approx_p_value  odds_ratio  p_adj_BH
                        intercept            -2.7840            0.2396  -11.6217          0.0000      0.0618       NaN
                         se_score            -0.7558            0.2072   -3.6475          0.0003      0.4696    0.0048
                               C5            -0.4735            0.1509   -3.1377          0.0017      0.6228    0.0094
D8G_Every day or almost every day             0.3769            0.1253    3.0089          0.0026      1.4577    0.0094
                  C1I_3. Disagree            -0.5050            0.2184   -2.3116          0.0208      0.6035    0.0416
   D37_PSOC_Personal care workers             0.3647     